In [3]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys 
from docx import Document
from docx.shared import Pt  # For font size
from docx.oxml.ns import qn  # For language customization
from docx.oxml import parse_xml  # For custom XML styling
from docx.shared import RGBColor  # For font color
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.oxml import OxmlElement
import random
import string
import pandas as pd
from pathlib import Path
import os

folder_path='G:/backup'
state=['' for i in range(1,432)]
social_security=['' for i in range(1,432)]


In [ ]:
chrome_options =  webdriver.ChromeOptions()
chrome_options.add_argument("--disable-gpu")  
chrome_options.add_argument("--blink-settings=imagesEnabled=false")  
chrome_options.add_experimental_option(
    "prefs", {"profile.managed_default_content_settings.images": 2,}
)
chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])
chrome_options.add_argument("--autoplay-policy=no-user-gesture-required")# Abort the request if it's a video file



In [1]:
def find_missing_pairs():
    # Get all files in directory
    files = os.listdir(folder_path)

    # Get all numbers from files
    elmi_numbers = set()
    gmat_numbers = set()

    for file in files:
        if file.endswith('.xlsx'):
            number = file.split('_')[0]
            if 'elmi' in file:
                elmi_numbers.add(number)
            elif 'gmat' in file:
                gmat_numbers.add(number)

    # Find numbers missing either file
    all_numbers = elmi_numbers | gmat_numbers
    for number in all_numbers:
        if number not in elmi_numbers or number not in gmat_numbers:
            print(number)

In [4]:
find_missing_pairs()

9353316543
9129380626


In [ ]:
def get_unique_phone_numbers():
    phone_numbers = set()

    # Scan the folder for files
    for file in os.listdir(folder_path):
        if file.endswith(".xlsx"):
            parts = file.split("_")
            if len(parts) >= 3 and parts[0].isdigit():  # Ensure valid format
                phone_numbers.add(parts[0])

    # Print unique numbers
    print("Unique Phone Numbers:")
    for number in sorted(phone_numbers):
        print(number)

    # Print count
    print(f"\nTotal Unique Numbers: {len(phone_numbers)}")

In [ ]:
get_unique_phone_numbers()

In [ ]:

def create_backup_list( output_file="backup-list.xlsx"):
    data_list = []

    # Scan folder for files
    files = os.listdir(folder_path)
    unique_numbers = set(f.split("_")[0] for f in files if f.endswith(".xlsx"))

    for number in unique_numbers:
        elmi_file = os.path.join(folder_path, f"{number}_elmi_result.xlsx")
        gmat_file = os.path.join(folder_path, f"{number}_gmat_result.xlsx")

        if os.path.exists(elmi_file) and os.path.exists(gmat_file):
            try:
                # Read data from _elmi_result.xlsx
                elmi_df = pd.read_excel(elmi_file, header=None)
                name = elmi_df.iloc[1, 0]  # Row 2, Column A (Name)
                phone_number = elmi_df.iloc[1, 1]  # Row 2, Column B (Number)
                elmi_score = elmi_df.iloc[1, 2]  # Row 2, Column C (Elmi Score)

                # Read data from _gmat_result.xlsx
                gmat_df = pd.read_excel(gmat_file, header=None)
                gmat_score = gmat_df.iloc[1, 2]  # Row 2, Column C (GMAT Score)

                # Store data
                data_list.append([name, phone_number, elmi_score, gmat_score])
            except Exception as e:
                print(f"Error processing {number}: {e}")

    # Create DataFrame for backup-list.xlsx
    backup_df = pd.DataFrame(data_list, columns=["Name", "Number", "Elmi", "GMAT"])

    # Save to Excel
    backup_df.to_excel(os.path.join(folder_path, output_file), index=False)

    print(f"Backup list saved as {output_file} in {folder_path}")


In [ ]:
# create_backup_list()


In [ ]:
service = Service("C:/Users/mehdi/Documents/webdrivers/chromedriver.exe")
global driver
driver = webdriver.Chrome(service=service, options=chrome_options)
driver.get('https://sjg.eadl.ir/accounts/login-pass/')

In [ ]:
def FindState(index):
    try:
        # Locate the province (state) field
        state_element = driver.find_element(By.CLASS_NAME, "province")
        state[index] = state_element.text.replace("استان:", "").strip()
        print('---------')
        print(f'state index:{index} is {state[index]}');
        print('---------')

        # Check if state is empty
# Assign a default value if empty

    except NoSuchElementException:
      print('')
      # Handle case where the element does not exist
    # Print the extracted state
    # print("State:", state)

In [ ]:
def Click():
    select=driver.find_element(By.XPATH,'//*[@id="root"]/div[1]/section/section/main/div/div/div/div[4]/div[2]/a/div/div')
    select.click()


In [ ]:
def FindSocial(index):
    try:
        # Locate the social security number field
        social_security_element = driver.find_element(By.XPATH, "//th[contains(span/text(), 'کد ملی:')]/span/label")
        print('social_security_element.text.strip()')
        print(social_security_element.text.strip())
        social_security[index] = social_security_element.text.strip()
        print
        print('---------')
        print(f'social_security index:{index} is {social_security[index]}');
        print('---------')
        # Check if the social security number is empty
   # Assign a default value if empty

    except NoSuchElementException:
        # social_security = "Not Available"  # Handle case where the element does not exist
        print('')
    # Print the extracted Social Security Number
    # print("Social Security Number:", social_security)

In [ ]:
df = pd.read_excel('backup-list.xlsx')

# Iterate through rows, starting from the second row (index 1) to skip the header
for index, row in df.iterrows():
  if pd.notna(row.iloc[1]):
    print('------------------------')
    print((index/430)*100)
    driver.get(f'https://sjg.eadl.ir/users/list?query_string={int(row[1])}&user=undefined')
    try:
      FindState(index=index)
    except Exception as e:
      print('')
      # print(f"Error in FindState for {row[1]}: {e}")
    
    try:
      Click()
    except Exception as e:
      print('')
      # print(f"Error in Click for {row[1]}: {e}")
    
    try:
      FindSocial(index=index)
    except Exception as e:
      print('')
      # print(f"Error in FindSocial for {row[1]}: {e}")
    
    # Check if Column B (index 1) has a phone number
    # df.iloc[index, 2] = social_security  # Update Column C (index 2)
    # df.iloc[index, 3] = state  # Update Column D (index 3)

# Save back to the same file
# df.to_excel('backup-list.xlsx', index=False)

In [ ]:
print(len(state))

In [ ]:
df.to_excel('backup-list.xlsx', index=False)

In [ ]:
with open("social_security.txt", "w", encoding="utf-8") as file:
    for item in social_security:
        file.write(f"{item}\n")

In [5]:
import openpyxl

def update_excel_with_state_data(text_file_path, excel_file_path, encoding="utf-8"):
    """
    Reads data from a text file containing Persian words and writes it into Column D
    of the specified Excel file with the header 'state'.
    The proper encoding should be used to correctly decode Persian characters.

    :param text_file_path: Path to the text file (e.g., state.txt).
    :param excel_file_path: Path to the Excel file (e.g., backup-list.xlsx).
    :param encoding: Encoding of the text file. Default is 'utf-8'.
                     Change to 'cp1256' or 'windows-1256' if needed.
    """
    # Load the existing workbook
    wb = openpyxl.load_workbook(excel_file_path)
    ws = wb.active  # Modify if you need a specific worksheet

    # Column D corresponds to column index 4.
    column_index = 4

    # Write header "state" in cell D1
    ws.cell(row=1, column=column_index, value="state")

    # Open and read the text file (state.txt) with the specified encoding
    with open(text_file_path, "r", encoding=encoding) as file:
        lines = file.readlines()

    # Write each line from the text file to Column D starting from row 2
    for i, line in enumerate(lines):
        row = i + 2  # Data starts from row 2 because row 1 is the header
        value = line.strip()  # Remove any whitespace/newline characters
        ws.cell(row=row, column=column_index, value=value)

    # Save the updated workbook
    wb.save(excel_file_path)
    print(f"Data from '{text_file_path}' has been successfully written to Column D of '{excel_file_path}' using encoding {encoding}.")

if __name__ == "__main__":
    # File paths (adjust if needed)
    text_file_path = "state.txt"
    excel_file_path = "backup-list.xlsx"

    # If your file uses UTF-8, use the default. Otherwise, try encoding="cp1256"
    update_excel_with_state_data(text_file_path, excel_file_path, encoding="utf-8")


Data from 'state.txt' has been successfully written to Column D of 'backup-list.xlsx' using encoding utf-8.
